In [8]:
!pip install ipywidgets
!pip install sqlalchemy
!pip install sqlite-utils
!pip install PyPDF2
!pip install streamlit PyPDF2
!pip install pytesseract
!apt-get install tesseract-ocr
!apt-get install tesseract-ocr-eng tesseract-ocr-hin tesseract-ocr-mar
!pip install python-dotenv



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 79.0 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr-eng is already the newest version (1:4.00~git30-7274cfa-1.1).
tesseract-ocr-eng set to manually installed.
The following NEW packages will be installed:
  tesseract-ocr-hin tesseract-ocr-mar
0 upgraded, 2 newly installed, 0 to remove and 41 not upgraded.
Need to get 1,775 kB 

In [9]:
!pip install streamlit
!pip install pyngrok


In [16]:
%%writefile app.py
import streamlit as st
import sqlite3
import pandas as pd

# -----------------------------
# PAGE CONFIG
# -----------------------------
st.set_page_config(
    page_title="StudyTrack AI",
    page_icon="📚",
    layout="wide"
)

# -----------------------------
# GLASSMORPHISM CSS
# -----------------------------
# -----------------------------
# UPDATED BLACK THEME + GLASSMORPHISM CSS
# -----------------------------
page_bg = """
<style>

/* REMOVE TOP WHITE SPACE */
[data-testid="stAppViewContainer"] {
    padding-top: 0rem !important;
}

/* FULL BLACK BACKGROUND */
body, html, [data-testid="stAppViewContainer"] {
    background-color: #000 !important;
    background: #000 !important;
}

/* REMOVE STREAMLIT HEADER */
header[data-testid="stHeader"] {
    background: rgba(0,0,0,0) !important;
    box-shadow: none !important;
    height: 0px !important;
}

/* SIDEBAR GLASS EFFECT */
section[data-testid="stSidebar"] {
    background: rgba(25, 25, 25, 0.55) !important;
    backdrop-filter: blur(18px);
}
section[data-testid="stSidebar"] div,
section[data-testid="stSidebar"] span,
section[data-testid="stSidebar"] label {
    color: #ffffff !important;     /* ← Sidebar text white */
}

/* MAIN GLASS CARD */
.glass-card {
    background: rgba(255, 255, 255, 0.08);
    padding: 25px;
    border-radius: 20px;
    border: 1px solid rgba(255,255,255,0.22);
    backdrop-filter: blur(15px);
    box-shadow: 0 8px 25px rgba(0,0,0,0.4);
}

/* BUTTON STYLE */
.stButton>button {
    background: linear-gradient(135deg, #8A2BE2, #C026D3);
    border-radius: 10px;
    border: none;
    color: white !important;
    padding: 12px 22px;
    font-size: 16px;
    font-weight: 600;
}

/* INPUT BOX FIX – remove black box + dark grey field */
.stTextInput>div>div>input,
.stNumberInput>div>div>input,
.stTextArea textarea {
    background: rgba(40,40,40,0.8) !important;
    color: white !important;
    border-radius: 10px;
    border: 1px solid rgba(255,255,255,0.2) !important;
}

/* Ensuring ALL labels and headings are white */
label,
p,
span,
h1, h2, h3, h4, h5, h6,
.stMarkdown,
.css-16idsys,
.css-10trblm {
    color: white !important;
}

/* TABS TEXT COLOR (View / Update / Delete Student) */
.stTabs [role="tab"] {
    color: white !important;
}
.stTabs [role="tab"][aria-selected="true"] {
    color: #C026D3 !important;        /* Highlight active tab */
    font-weight: 700 !important;
}

</style>
"""
st.markdown(page_bg, unsafe_allow_html=True)


# -----------------------------
# DB CONNECTION
# -----------------------------
conn = sqlite3.connect("studytrack.db", check_same_thread=False)
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS students (
    student_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    email TEXT UNIQUE,
    password TEXT
);
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS study_logs (
    log_id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_email TEXT,
    subject TEXT,
    hours INTEGER,
    date TEXT
);
""")

conn.commit()

# -----------------------------
# SESSION STATE
# -----------------------------
if "page" not in st.session_state:
    st.session_state["page"] = "home"

if "logged_email" not in st.session_state:
    st.session_state["logged_email"] = None

# -----------------------------
# FUNCTIONS
# -----------------------------
def register_student(name, email, password):
    try:
        cur.execute("INSERT INTO students (name, email, password) VALUES (?, ?, ?)",
                    (name, email, password))
        conn.commit()
        return True
    except:
        return False

def check_student_login(email, password):
    cur.execute("SELECT * FROM students WHERE email=? AND password=?", (email, password))
    return cur.fetchone()

def save_log(email, subject, hours, date):
    cur.execute("INSERT INTO study_logs (student_email, subject, hours, date) VALUES (?, ?, ?, ?)",
                (email, subject, hours, date))
    conn.commit()

def get_student_logs(email):
    df = pd.read_sql_query(f"SELECT * FROM study_logs WHERE student_email='{email}'", conn)
    return df

# -----------------------------
# ADMIN
# -----------------------------
ADMIN_USER = "sneha"
ADMIN_PASS = "1234"

def admin_login(user, password):
    return user == ADMIN_USER and password == ADMIN_PASS

def admin_get_students():
    return pd.read_sql_query("SELECT * FROM students", conn)

def admin_delete_student(email):
    cur.execute("DELETE FROM students WHERE email=?", (email,))
    conn.commit()

def admin_update_student(email, new_name):
    cur.execute("UPDATE students SET name=? WHERE email=?", (new_name, email))
    conn.commit()


# ======================================================
# SIDEBAR MENU
# ======================================================
st.sidebar.title("📌 Navigation")
menu = st.sidebar.radio(
    "Choose:",
    ["Student Login", "Student Register", "Admin Login"]
)


# ======================================================
# HOME PAGE (Login + Register Glass Cards)
# ======================================================
if st.session_state["page"] == "home":

    st.markdown("<h1 style='color:white;text-align:center;'>📚 StudyTrack AI</h1>", unsafe_allow_html=True)
    st.markdown("<h4 style='color:white;text-align:center;'>Smart Study Habit Recommendation System</h4>", unsafe_allow_html=True)
    st.write("")

    st.markdown("<div class='glass-card'>", unsafe_allow_html=True)

    # Student Register
    if menu == "Student Register":
        st.subheader("📝 Create Student Account")
        name = st.text_input("Full Name")
        email = st.text_input("Email")
        password = st.text_input("Password", type="password")

        if st.button("Register"):
            if register_student(name, email, password):
                st.success("🎉 Registration Successful!")
            else:
                st.error("Email already exists!")

    # Student Login
    elif menu == "Student Login":
        st.subheader("🔐 Student Login")
        email = st.text_input("Email")
        password = st.text_input("Password", type="password")

        if st.button("Login"):
            student = check_student_login(email, password)
            if student:
                st.session_state["logged_email"] = email
                st.session_state["page"] = "student_dashboard"
                st.rerun()
            else:
                st.error("❌ Invalid Login Details")

    # Admin Login
    elif menu == "Admin Login":
        st.subheader("🛠 Admin Login")
        username = st.text_input("Admin Username")
        password = st.text_input("Admin Password", type="password")

        if st.button("Admin Login"):
            if admin_login(username, password):
                st.session_state["page"] = "admin_dashboard"
                st.rerun()
            else:
                st.error("❌ Invalid Admin Credentials")

    st.markdown("</div>", unsafe_allow_html=True)



# ======================================================
# STUDENT DASHBOARD
# ======================================================
elif st.session_state["page"] == "student_dashboard":

    st.markdown("<h1 style='color:white;'>🎓 Student Dashboard</h1>", unsafe_allow_html=True)
    st.success(f"Logged in as: {st.session_state['logged_email']}")

    tab1, tab2 = st.tabs(["➕ Add Study Log", "📄 My Study Logs"])

    # Add Study Log
    with tab1:
        st.markdown("<div class='glass-card'>", unsafe_allow_html=True)
        subject = st.text_input("Subject")
        hours = st.number_input("Hours Studied", min_value=0)
        date = st.date_input("Date")

        if st.button("Save Log"):
            save_log(st.session_state["logged_email"], subject, hours, str(date))
            st.success("📌 Study Log Saved!")
        st.markdown("</div>", unsafe_allow_html=True)

    # View Logs
    with tab2:
        st.markdown("<div class='glass-card'>", unsafe_allow_html=True)
        df = get_student_logs(st.session_state["logged_email"])
        st.dataframe(df)
        st.markdown("</div>", unsafe_allow_html=True)

    if st.button("Logout"):
        st.session_state["page"] = "home"
        st.rerun()



# ======================================================
# ADMIN DASHBOARD
# ======================================================
elif st.session_state["page"] == "admin_dashboard":

    st.markdown("<h1 style='color:white;'>🛠 Admin Panel</h1>", unsafe_allow_html=True)

    tab1, tab2, tab3 = st.tabs([
        "📋 View Students",
        "✏ Update Student",
        "🗑 Delete Student"
    ])

    # VIEW STUDENTS
    with tab1:
        st.markdown("<div class='glass-card'>", unsafe_allow_html=True)
        df = admin_get_students()
        st.dataframe(df)
        st.markdown("</div>", unsafe_allow_html=True)

    # UPDATE STUDENTS
    with tab2:
        st.markdown("<div class='glass-card'>", unsafe_allow_html=True)
        update_email = st.text_input("Email to Update")
        new_name = st.text_input("New Name")
        if st.button("Update"):
            admin_update_student(update_email, new_name)
            st.success("✔ Student Updated!")
        st.markdown("</div>", unsafe_allow_html=True)

    # DELETE STUDENTS
    with tab3:
        st.markdown("<div class='glass-card'>", unsafe_allow_html=True)
        del_email = st.text_input("Email to Delete")
        if st.button("Delete"):
            admin_delete_student(del_email)
            st.success("🗑 Student Deleted!")
        st.markdown("</div>", unsafe_allow_html=True)

    if st.button("Logout"):
        st.session_state["page"] = "home"
        st.rerun()

Overwriting app.py
Overwriting app.py


In [11]:
!streamlit run app.py &>/dev/null &

In [12]:
# FULL NGROK RESET
!rm -rf /root/.ngrok2
!rm -rf /root/.config/ngrok
!rm -rf ~/.ngrok2
!rm -rf ~/.config/ngrok

from pyngrok import ngrok
ngrok.kill()

In [13]:
!ngrok config add-authtoken 34x7BDwOLuMqF2m9bJHquhAnRWo_59ABPzGXeHNoRT2SR4FiN

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [14]:
from pyngrok import ngrok
ngrok.kill()

public_url = ngrok.connect(8501)
public_url

<NgrokTunnel: "https://margert-ablest-undespondingly.ngrok-free.dev" -> "http://localhost:8501">